# [Hate Speech Identification Shared Task](https://multihate.github.io/): Subtask 1A at [BLP Workshop](https://blp-workshop.github.io/) @IJCNLP-AACL 2025

This shared task is designed to identify the type of hate, its severity, and the targeted group from social media content. The goal is to develop robust systems that advance research in this area.

In this subtask, given a Bangla text collected from YouTube comments, categorize whether it contains abusive, sexism, religious hate, political hate, profane, or none.

### Downloading dataset from github

In [14]:
!wget https://raw.githubusercontent.com/AridHasan/blp25_task1/refs/heads/main/data/subtask_1A/blp25_hatespeech_subtask_1A_train.tsv
!wget https://raw.githubusercontent.com/AridHasan/blp25_task1/refs/heads/main/data/subtask_1A/blp25_hatespeech_subtask_1A_dev.tsv
!wget https://raw.githubusercontent.com/AridHasan/blp25_task1/refs/heads/main/data/subtask_1A/blp25_hatespeech_subtask_1A_dev_test.tsv

--2025-10-01 12:04:56--  https://raw.githubusercontent.com/AridHasan/blp25_task1/refs/heads/main/data/subtask_1A/blp25_hatespeech_subtask_1A_train.tsv
Resolving raw.githubusercontent.com (raw.githubusercontent.com)... 185.199.109.133, 185.199.111.133, 185.199.110.133, ...
Connecting to raw.githubusercontent.com (raw.githubusercontent.com)|185.199.109.133|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 8002036 (7.6M) [text/plain]
Saving to: ‘blp25_hatespeech_subtask_1A_train.tsv.1’

blp25_hatespeech_su 100%[===================>]   7.63M  --.-KB/s    in 0.09s   

2025-10-01 12:04:56 (88.2 MB/s) - ‘blp25_hatespeech_subtask_1A_train.tsv.1’ saved [8002036/8002036]

--2025-10-01 12:04:56--  https://raw.githubusercontent.com/AridHasan/blp25_task1/refs/heads/main/data/subtask_1A/blp25_hatespeech_subtask_1A_dev.tsv
Resolving raw.githubusercontent.com (raw.githubusercontent.com)... 185.199.111.133, 185.199.108.133, 185.199.110.133, ...
Connecting to raw.githubuserconten

### installing required libraries.
 - transformers
 - datasets
 - evaluate
 - accelerate

In [15]:
!pip install transformers
!pip install datasets
!pip install evaluate
# !pip install --upgrade accelerate

#### importing required libraries and setting up logger

In [16]:
import logging
import os
import random
import sys
from dataclasses import dataclass, field
from typing import Optional
import pandas as pd
import datasets
import evaluate
import numpy as np
from datasets import load_dataset, Dataset, DatasetDict
import torch

import transformers
from transformers import (
    AutoConfig,
    AutoModelForSequenceClassification,
    AutoTokenizer,
    DataCollatorWithPadding,
    EvalPrediction,
    HfArgumentParser,
    PretrainedConfig,
    Trainer,
    TrainingArguments,
    default_data_collator,
    set_seed,
)
from transformers.trainer_utils import get_last_checkpoint
from transformers.utils import check_min_version, send_example_telemetry
from transformers.utils.versions import require_version


logger = logging.getLogger(__name__)

logging.basicConfig(
    format="%(asctime)s - %(levelname)s - %(name)s - %(message)s",
    datefmt="%m/%d/%Y %H:%M:%S",
    handlers=[logging.StreamHandler(sys.stdout)],
)

### Defining the training, validation, and test data

In [17]:
train_file = "/kaggle/input/new-dataset-v5/label_hate_severity_train.tsv"
validation_file = '/kaggle/input/new-dataset-v5/label_hate_severity_validation.tsv'
test_file = '/kaggle/input/new-dataset-v5/blp25_hatespeech_subtask_1C_test.tsv'

### Disable wandb

In [18]:
import os
os.environ["WANDB_DISABLED"] = "true"

In [19]:
print(transformers.__version__)

4.52.4


In [20]:
# from transformers import TrainingArguments
# print("Using:", TrainingArguments.__module__, TrainingArguments.__name__)

In [21]:
# TA = transformers.TrainingArguments
# ta_fields = {f.name for f in fields(TA)}
# print(ta_fields)

### Setting up the training parameters

In [22]:
# training_args = TrainingArguments(
#     learning_rate=2e-5,
#     num_train_epochs=3,
#     per_device_train_batch_size=16,
#     per_device_eval_batch_size=16,
#     output_dir="./distilBERT_m/",
#     overwrite_output_dir=True,
#     remove_unused_columns=False,
#     #local_rank= 1,
#     load_best_model_at_end=True,
#     save_total_limit=2,
#     save_strategy="epoch",
#     evaluation_strategy="epoch",
#     report_to=None,
#     warmup_ratio=0.06,
#     weight_decay=0.01,
#     logging_strategy="steps",
#     logging_steps=50,
# )
training_args = transformers.TrainingArguments(
    output_dir="./distilBERT_m/",
    overwrite_output_dir=True,
    learning_rate=2e-5,
    num_train_epochs=5,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=16,
    eval_strategy="epoch",
    save_strategy="epoch",
    load_best_model_at_end=True,
    metric_for_best_model="accuracy",
    greater_is_better=True,
    warmup_ratio=0.06,
    weight_decay=0.01,
    logging_strategy="steps",
    logging_steps=50,
    report_to=None,
    remove_unused_columns=False
)
max_train_samples = None
max_eval_samples=None
max_predict_samples=None
max_seq_length = 512
batch_size = 16

[INFO|training_args.py:2135] 2025-10-01 12:05:10,203 >> PyTorch: setting up devices
[INFO|training_args.py:1812] 2025-10-01 12:05:10,206 >> The default value for the training argument `--report_to` will change in v5 (from all installed integrations to none). In v5, you will need to use `--report_to all` to get the same behavior as now. You should start updating your code and make this info disappear :-).
[WARNING|integration_utils.py:101] 2025-10-01 12:05:10,210 >> Using the `WANDB_DISABLED` environment variable is deprecated and will be removed in v5. Use the --report_to flag to control the integrations used for logging result (for instance --report_to none).


In [23]:
transformers.utils.logging.set_verbosity_info()

log_level = training_args.get_process_log_level()
logger.setLevel(log_level)
datasets.utils.logging.set_verbosity(log_level)
transformers.utils.logging.set_verbosity(log_level)
transformers.utils.logging.enable_default_handler()
transformers.utils.logging.enable_explicit_format()
logger.warning(
    f"Process rank: {training_args.local_rank}, device: {training_args.device}, n_gpu: {training_args.n_gpu}"
    + f" distributed training: {bool(training_args.local_rank != -1)}, 16-bits training: {training_args.fp16}"
)
logger.info(f"Training/evaluation parameters {training_args}")

#### Defining the Model

In [24]:
#model_name = 'distilbert-base-multilingual-cased'
#model_name = 'csebuetnlp/banglabert'
#model_name = 'csebuetnlp/banglabert'
#model_name = 'csebuetnlp/banglabert'
#model_name = 'cardiffnlp/twitter-xlm-roberta-base-sentiment'
#model_name =  'intfloat/multilingual-e5-base'
model_name = 'google/muril-base-cased'

#### setting the random seed

In [25]:
set_seed(training_args.seed)

#### Loading data files

In [26]:
#l2id = {'None': 0, 'Religious Hate': 1, 'Sexism': 2, 'Political Hate': 3, 'Profane': 4, 'Abusive': 5}
l2id = {
    "Severe" : 0,
    "Little to None" : 1,
    "Mild" : 2
}
train_df = pd.read_csv(train_file, sep='\t',dtype=str, keep_default_na=False)
# print(train_df['label'])
train_df['label'] = train_df['label'].map(l2id).fillna(0).astype(int)
train_df = Dataset.from_pandas(train_df)
validation_df = pd.read_csv(validation_file, sep='\t',dtype=str, keep_default_na=False)
validation_df['label'] = validation_df['label'].map(l2id).fillna(0).astype(int)
validation_df = Dataset.from_pandas(validation_df)
test_df = pd.read_csv(test_file, sep='\t',dtype=str, keep_default_na=False)
#test_df['label'] = test_df['label'].map(l2id)
test_df = Dataset.from_pandas(test_df)

data_files = {"train": train_df, "validation": validation_df, "test": test_df}
for key in data_files.keys():
    logger.info(f"loading a local file for {key}")
raw_datasets = DatasetDict(
    {"train": train_df, "validation": validation_df, "test": test_df}
)

non_label_column_names = [name for name in raw_datasets["train"].column_names if name != "label"]
sentence1_key= non_label_column_names[1]

# Padding strategy
padding = "max_length"

label_list = ["Severe","Little to None","Mild"]

#l2id = {'None': 0, 'Religious Hate': 1, 'Sexism': 2, 'Political Hate': 3, 'Profane': 4, 'Abusive': 5}
id2label = {v: k for k, v in l2id.items()}



##### Extracting number of unique labels

In [27]:
# Labels
label_list = raw_datasets["train"].unique("label")
print(label_list)
label_list.sort()  # sort the labels for determine
num_labels = len(label_list)

[1, 0, 2]


### Loading Pretrained Configuration, Tokenizer and Model

In [28]:
config = AutoConfig.from_pretrained(
    model_name,
    num_labels=num_labels,
    finetuning_task=None,
    cache_dir=None,
    revision="main",
    use_auth_token=None,
)
#config.num_labels = 6

tokenizer = AutoTokenizer.from_pretrained(
    model_name,
    cache_dir=None,
    use_fast=True,
    revision="main",
    use_auth_token=None,
)

model = AutoModelForSequenceClassification.from_pretrained(
    model_name,
    from_tf=bool(".ckpt" in model_name),
    config=config,
    cache_dir=None,
    revision="main",
    use_auth_token=None,
    ignore_mismatched_sizes=True,
)

config.json:   0%|          | 0.00/411 [00:00<?, ?B/s]

[INFO|configuration_utils.py:698] 2025-10-01 12:05:11,142 >> loading configuration file config.json from cache at /root/.cache/huggingface/hub/models--google--muril-base-cased/snapshots/afd9f36c7923d54e97903922ff1b260d091d202f/config.json
[INFO|configuration_utils.py:770] 2025-10-01 12:05:11,147 >> Model config BertConfig {
  "architectures": [
    "BertForMaskedLM"
  ],
  "attention_probs_dropout_prob": 0.1,
  "classifier_dropout": null,
  "embedding_size": 768,
  "hidden_act": "gelu",
  "hidden_dropout_prob": 0.1,
  "hidden_size": 768,
  "id2label": {
    "0": "LABEL_0",
    "1": "LABEL_1",
    "2": "LABEL_2"
  },
  "initializer_range": 0.02,
  "intermediate_size": 3072,
  "label2id": {
    "LABEL_0": 0,
    "LABEL_1": 1,
    "LABEL_2": 2
  },
  "layer_norm_eps": 1e-12,
  "max_position_embeddings": 512,
  "model_type": "bert",
  "num_attention_heads": 12,
  "num_hidden_layers": 12,
  "pad_token_id": 0,
  "position_embedding_type": "absolute",
  "transformers_version": "4.52.4",
  "ty

tokenizer_config.json:   0%|          | 0.00/206 [00:00<?, ?B/s]

[INFO|configuration_utils.py:698] 2025-10-01 12:05:11,295 >> loading configuration file config.json from cache at /root/.cache/huggingface/hub/models--google--muril-base-cased/snapshots/afd9f36c7923d54e97903922ff1b260d091d202f/config.json
[INFO|configuration_utils.py:770] 2025-10-01 12:05:11,296 >> Model config BertConfig {
  "architectures": [
    "BertForMaskedLM"
  ],
  "attention_probs_dropout_prob": 0.1,
  "classifier_dropout": null,
  "embedding_size": 768,
  "hidden_act": "gelu",
  "hidden_dropout_prob": 0.1,
  "hidden_size": 768,
  "initializer_range": 0.02,
  "intermediate_size": 3072,
  "layer_norm_eps": 1e-12,
  "max_position_embeddings": 512,
  "model_type": "bert",
  "num_attention_heads": 12,
  "num_hidden_layers": 12,
  "pad_token_id": 0,
  "position_embedding_type": "absolute",
  "transformers_version": "4.52.4",
  "type_vocab_size": 2,
  "use_cache": true,
  "vocab_size": 197285
}



vocab.txt: 0.00B [00:00, ?B/s]

...

[INFO|tokenization_utils_base.py:2023] 2025-10-01 12:05:11,795 >> loading file vocab.txt from cache at /root/.cache/huggingface/hub/models--google--muril-base-cased/snapshots/afd9f36c7923d54e97903922ff1b260d091d202f/vocab.txt
[INFO|tokenization_utils_base.py:2023] 2025-10-01 12:05:11,796 >> loading file tokenizer.json from cache at None
[INFO|tokenization_utils_base.py:2023] 2025-10-01 12:05:11,797 >> loading file added_tokens.json from cache at None
[INFO|tokenization_utils_base.py:2023] 2025-10-01 12:05:11,798 >> loading file special_tokens_map.json from cache at /root/.cache/huggingface/hub/models--google--muril-base-cased/snapshots/afd9f36c7923d54e97903922ff1b260d091d202f/special_tokens_map.json
[INFO|tokenization_utils_base.py:2023] 2025-10-01 12:05:11,799 >> loading file tokenizer_config.json from cache at /root/.cache/huggingface/hub/models--google--muril-base-cased/snapshots/afd9f36c7923d54e97903922ff1b260d091d202f/tokenizer_config.json
[INFO|tokenization_utils_base.py:2023] 20

pytorch_model.bin:   0%|          | 0.00/953M [00:00<?, ?B/s]

[INFO|modeling_utils.py:1151] 2025-10-01 12:05:20,280 >> loading weights file pytorch_model.bin from cache at /root/.cache/huggingface/hub/models--google--muril-base-cased/snapshots/afd9f36c7923d54e97903922ff1b260d091d202f/pytorch_model.bin
[INFO|safetensors_conversion.py:61] 2025-10-01 12:05:20,387 >> Attempting to create safetensors variant
[INFO|safetensors_conversion.py:74] 2025-10-01 12:05:20,588 >> Safetensors PR exists


model.safetensors:   0%|          | 0.00/953M [00:00<?, ?B/s]

[INFO|modeling_utils.py:5121] 2025-10-01 12:05:28,884 >> Some weights of the model checkpoint at google/muril-base-cased were not used when initializing BertForSequenceClassification: ['cls.predictions.bias', 'cls.predictions.decoder.bias', 'cls.predictions.decoder.weight', 'cls.predictions.transform.LayerNorm.bias', 'cls.predictions.transform.LayerNorm.weight', 'cls.predictions.transform.dense.bias', 'cls.predictions.transform.dense.weight', 'cls.seq_relationship.bias', 'cls.seq_relationship.weight']
- This IS expected if you are initializing BertForSequenceClassification from the checkpoint of a model trained on another task or with another architecture (e.g. initializing a BertForSequenceClassification model from a BertForPreTraining model).
- This IS NOT expected if you are initializing BertForSequenceClassification from the checkpoint of a model that you expect to be exactly identical (initializing a BertForSequenceClassification model from a BertForSequenceClassification model).


#### Preprocessing the raw_datasets

In [29]:
# non_label_column_names = [name for name in raw_datasets["train"].column_names if name != "label"]
# sentence1_key= non_label_column_names[1]

# # Padding strategy
# padding = "max_length"

# # Some models have set the order of the labels to use, so let's make sure we do use it.
# label_to_id = None
# if (model.config.label2id != PretrainedConfig(num_labels=num_labels).label2id):
#     # Some have all caps in their config, some don't.
#     label_name_to_id = {k.lower(): v for k, v in model.config.label2id.items()}
#     if sorted(label_name_to_id.keys()) == sorted(label_list):
#         label_to_id = {i: int(label_name_to_id[label_list[i]]) for i in range(num_labels)}
#     else:
#         logger.warning(
#             "Your model seems to have been trained with labels, but they don't match the dataset: ",
#             f"model labels: {sorted(label_name_to_id.keys())}, dataset labels: {sorted(label_list)}."
#             "\nIgnoring the model labels as a result.",)

# if label_to_id is not None:
#     model.config.label2id = label_to_id
#     model.config.id2label = {id: label for label, id in config.label2id.items()}

# if 128 > tokenizer.model_max_length:
#     logger.warning(
#         f"The max_seq_length passed ({128}) is larger than the maximum length for the"
#         f"model ({tokenizer.model_max_length}). Using max_seq_length={tokenizer.model_max_length}.")
# max_seq_length = min(128, tokenizer.model_max_length)

# def preprocess_function(examples):
#     # Tokenize the texts
#     args = (
#         (examples[sentence1_key],))
#     result = tokenizer(*args, padding=padding, max_length=max_seq_length, truncation=True)

#     # Map labels to IDs (not necessary for GLUE tasks)
#     if label_to_id is not None and "label" in examples:
#         result["label"] = [(label_to_id[l] if l != -1 else -1) for l in examples["label"]]
#     return result
# raw_datasets = raw_datasets.map(
#     preprocess_function,
#     batched=True,
#     load_from_cache_file=True,
#     desc="Running tokenizer on dataset",
# )


#### Finalize the training data for training the model

In [30]:
model.config.label2id = l2id
model.config.id2label = id2label

if hasattr(model, "num_labels"):
    model.num_labels = num_labels

if 128 > tokenizer.model_max_length:
    logger.warning(
        f"The max_seq_length passed ({128}) is larger than the maximum length for the"
        f"model ({tokenizer.model_max_length}). Using max_seq_length={tokenizer.model_max_length}.")
max_seq_length = min(128, tokenizer.model_max_length)

def preprocess_function(examples):
    # Tokenize the texts
    args = (
        (examples[sentence1_key],))
    result = tokenizer(*args, padding=padding, max_length=max_seq_length, truncation=True)

    if "label" in examples:
        result["label"] = examples["label"] 
    return result
raw_datasets = raw_datasets.map(
    preprocess_function,
    batched=True,
    load_from_cache_file=True,
    desc="Running tokenizer on dataset",
)


Running tokenizer on dataset:   0%|          | 0/35522 [00:00<?, ? examples/s]

Running tokenizer on dataset:   0%|          | 0/2512 [00:00<?, ? examples/s]

Running tokenizer on dataset:   0%|          | 0/10200 [00:00<?, ? examples/s]

In [31]:
print(raw_datasets)

DatasetDict({
    train: Dataset({
        features: ['id', 'text', 'label', 'input_ids', 'token_type_ids', 'attention_mask'],
        num_rows: 35522
    })
    validation: Dataset({
        features: ['id', 'text', 'label', 'input_ids', 'token_type_ids', 'attention_mask'],
        num_rows: 2512
    })
    test: Dataset({
        features: ['id', 'text', 'input_ids', 'token_type_ids', 'attention_mask'],
        num_rows: 10200
    })
})


In [32]:
# if "train" not in raw_datasets:
#     raise ValueError("requires a train dataset")
# train_dataset = raw_datasets["train"]
# if max_train_samples is not None:
#     max_train_samples_n = min(len(train_dataset), max_train_samples)
#     train_dataset = train_dataset.select(range(max_train_samples_n))

In [33]:
if "train" not in raw_datasets:
    raise ValueError("requires a train dataset")
train_dataset = raw_datasets["train"]
max_train_samples_n = len(train_dataset)


In [34]:
train_dataset

Dataset({
    features: ['id', 'text', 'label', 'input_ids', 'token_type_ids', 'attention_mask'],
    num_rows: 35522
})

#### Finalize the development/evaluation data for evaluating the model

In [35]:
# if "validation" not in raw_datasets:
#     raise ValueError("requires a validation dataset")
# eval_dataset = raw_datasets["validation"]
# if max_eval_samples is not None:
#     max_eval_samples_n = min(len(eval_dataset), max_eval_samples)
#     eval_dataset = eval_dataset.select(range(max_eval_samples_n))

In [36]:
if "validation" not in raw_datasets:
    raise ValueError("requires a validation dataset")
eval_dataset = raw_datasets["validation"]
max_eval_samples_n = len(eval_dataset)

#### Finalize the test data for predicting the unseen test data using the model

In [37]:
# if "test" not in raw_datasets and "test_matched" not in raw_datasets:
#     raise ValueError("requires a test dataset")
# predict_dataset = raw_datasets["test"]
# if max_predict_samples is not None:
#     max_predict_samples_n = min(len(predict_dataset), max_predict_samples)
#     predict_dataset = predict_dataset.select(range(max_predict_samples_n))

In [38]:
predict_dataset = raw_datasets["test"]
max_predict_samples_n = len(predict_dataset)

#### Log a few random samples from the training set

In [39]:
for index in random.sample(range(len(train_dataset)), 3):
    logger.info(f"Sample {index} of the training set: {train_dataset[index]}.")

#### Get the metric function `accuracy`

In [40]:
metric = evaluate.load("accuracy")

#### Predictions and label_ids field and has to return a dictionary string to float.

In [41]:
def compute_metrics(p: EvalPrediction):
    preds = p.predictions[0] if isinstance(p.predictions, tuple) else p.predictions
    preds = np.argmax(preds, axis=1)
    return {"accuracy": (preds == p.label_ids).astype(np.float32).mean().item()}


#### Data Collator

In [42]:
data_collator = default_data_collator

#### Initialize our Trainer

In [43]:
train_dataset = train_dataset.remove_columns("id")
eval_dataset = eval_dataset.remove_columns("id")

In [44]:
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=eval_dataset,
    compute_metrics=compute_metrics,
    tokenizer=tokenizer,
    data_collator=data_collator,
)

/tmp/ipykernel_36/1049306949.py:1: FutureWarning: `tokenizer` is deprecated and will be removed in version 5.0.0 for `Trainer.__init__`. Use `processing_class` instead.
  trainer = Trainer(


#### Training our model

In [45]:
# train_result = trainer.train()
# metrics = train_result.metrics
# max_train_samples = (
#     max_train_samples if max_train_samples is not None else len(train_dataset)
# )
# metrics["train_samples"] = min(max_train_samples, len(train_dataset))



In [ ]:
train_result = trainer.train()
metrics = train_result.metrics
metrics["train_samples"] = len(train_dataset)

# 

[INFO|trainer.py:2409] 2025-10-01 12:05:54,364 >> ***** Running training *****
[INFO|trainer.py:2410] 2025-10-01 12:05:54,366 >>   Num examples = 35,522
[INFO|trainer.py:2411] 2025-10-01 12:05:54,371 >>   Num Epochs = 5
[INFO|trainer.py:2412] 2025-10-01 12:05:54,371 >>   Instantaneous batch size per device = 16
[INFO|trainer.py:2415] 2025-10-01 12:05:54,373 >>   Total train batch size (w. parallel, distributed & accumulation) = 16
[INFO|trainer.py:2416] 2025-10-01 12:05:54,375 >>   Gradient Accumulation steps = 1
[INFO|trainer.py:2417] 2025-10-01 12:05:54,376 >>   Total optimization steps = 11,105
[INFO|trainer.py:2418] 2025-10-01 12:05:54,380 >>   Number of trainable parameters = 237,558,531


Epoch,Training Loss,Validation Loss


#### Saving the tokenizer too for easy upload

In [ ]:
# trainer.save_model()
# trainer.log_metrics("train", metrics)
# trainer.save_metrics("train", metrics)
# trainer.save_state()

#### Evaluating our model on validation/development data

In [ ]:
# logger.info("*** Evaluate ***")

# metrics = trainer.evaluate(eval_dataset=eval_dataset)

# max_eval_samples = (
#     max_eval_samples if max_eval_samples is not None else len(eval_dataset)
# )
# metrics["eval_samples"] = min(max_eval_samples, len(eval_dataset))

# trainer.log_metrics("eval", metrics)
# trainer.save_metrics("eval", metrics)

In [ ]:
logger.info("*** Evaluate ***")

metrics = trainer.evaluate(eval_dataset=eval_dataset)

metrics["eval_samples"] = len(eval_dataset)

trainer.log_metrics("eval", metrics)
trainer.save_metrics("eval", metrics)

### Predecting the test data

In [ ]:
# id2l = {v: k for k, v in l2id.items()}
# logger.info("*** Predict ***")
# #predict_dataset = predict_dataset.remove_columns("label")
# ids = predict_dataset['id']
# predict_dataset = predict_dataset.remove_columns("id")
# predictions = trainer.predict(predict_dataset, metric_key_prefix="predict").predictions
# predictions = np.argmax(predictions, axis=1)
# output_predict_file = os.path.join(training_args.output_dir, f"subtask_1A.tsv")
# if trainer.is_world_process_zero():
#     with open(output_predict_file, "w") as writer:
#         logger.info(f"***** Predict results *****")
#         writer.write("id\tlabel\tmodel\n")
#         for index, item in enumerate(predictions):
#             item = label_list[item]
#             item = id2l[item]
#             writer.write(f"{ids[index]}\t{item}\t{model_name}\n")

In [ ]:
id2l = {v: k for k, v in l2id.items()}
logger.info("*** Predict ***")
#predict_dataset = predict_dataset.remove_columns("label")
ids = predict_dataset['id']
predict_dataset = predict_dataset.remove_columns(["id"])
predictions = trainer.predict(predict_dataset).predictions
predictions = np.argmax(predictions, axis=1)
output_predict_file = os.path.join(training_args.output_dir, f"subtask_1A.tsv")
if trainer.is_world_process_zero():
    with open(output_predict_file, "w") as writer:
        logger.info(f"***** Predict results *****")
        writer.write("id\tlabel\tmodel\n")
        for index, item in enumerate(predictions):
            item = id2l[item]
            writer.write(f"{ids[index]}\t{item}\t{model_name}\n")

In [ ]:
ids[0]

#### Saving the model into card

In [ ]:
# kwargs = {"finetuned_from": model_name, "tasks": "text-classification"}
# trainer.create_model_card(**kwargs)

In [ ]:
#!zip subtask_1A.zip ./distilBERT_m/subtask_1A.tsv

In [ ]:
# from sklearn.metrics import accuracy_score, classification_report
# import numpy as np

# def evaluate_on_test(trainer, test_dataset, id2label=None):
#     out = trainer.predict(test_dataset)
#     preds = np.argmax(out.predictions, axis=1)
#     labels = out.label_ids
#     acc = accuracy_score(labels, preds)
#     print(f"Test Accuracy: {acc:.4f}")
#     print(classification_report(labels, preds))
#     return {"accuracy": acc, "preds": preds, "labels": labels}

# # Example flow:
# # best_acc = train_select_best(trainer, train_dataset, eval_dataset, num_epochs=3)
# # test_metrics = evaluate_on_test(trainer, predict_dataset, id2label=model.config.id2label)
# test_metrics = evaluate_on_test(
#     trainer=trainer,
#     test_dataset=predict_dataset,
#     id2label=model.config.id2label
# )
# print(test_metrics)

In [ ]:
out_dir = "/kaggle/working/finetuned_blp_model"

# Save model + tokenizer
# trainer.save_model(out_dir)         # saves model + config
# tokenizer.save_pretrained(out_dir)  # saves tokenizer

# print(f"Saved to: {out_dir}")

In [ ]:
# from transformers import AutoModelForSequenceClassification, AutoTokenizer

# model = AutoModelForSequenceClassification.from_pretrained(out_dir)
# tokenizer = AutoTokenizer.from_pretrained(out_dir)